# Transfer Learning for image data using CNN

In [ ]:
# Install dependencies 
!pip install kaggle torch torchvision scikit-learn matplotlib > /dev/null 2>&1

## Kaggle Setup, Download, and Sorting

In [ ]:

import os
import shutil

# 1. setting up kaggle credentials
os.environ['KAGGLE_USERNAME'] = "abhishek072891"
os.environ['KAGGLE_KEY'] = "KGAT_ca4c8d4bd81203c895e1f5768bac947d"

# 2. Clean up any old files to ensure a completely fresh start
print("Wiping old directories...")
!rm -rf dataset raw_dataset chicken-duck-dataset.zip

# 3. Download and unzip the Kaggle dataset
print("Downloading dataset from Kaggle...")
!kaggle datasets download -d nothingnone123/chicken-duck-dataset
print("Unzipping dataset...")
!unzip -q chicken-duck-dataset.zip -d raw_dataset

# 4. Create fresh PyTorch-compatible target folders
print("Creating target folders...")
os.makedirs('dataset/chicken', exist_ok=True)
os.makedirs('dataset/duck', exist_ok=True)

# 5. Hunt down and correctly sort the images
print("Sorting images...")
c_count = 0
d_count = 0

for root, dirs, files_list in os.walk('raw_dataset'):
    for file in files_list:
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
            src_path = os.path.join(root, file)

            # Only look at the immediate parent folder name to prevent mix-ups
            parent_folder = os.path.basename(root).lower()

            if 'chicken' in parent_folder:
                shutil.copy(src_path, os.path.join('dataset/chicken', f'c_{c_count}.jpg'))
                c_count += 1
            elif 'duck' in parent_folder:
                shutil.copy(src_path, os.path.join('dataset/duck', f'd_{d_count}.jpg'))
                d_count += 1

print("\n=== SYSTEM REPORT ===")
print(f"Chicken images loaded: {c_count}")
print(f"Duck images loaded: {d_count}")
print("SUCCESS! You are ready to run Cell 3 (Data Loading).")

Wiping old directories...
Dataset URL: https://www.kaggle.com/datasets/nothingnone123/chicken-duck-dataset
License(s): MIT
100% 22.5M/22.5M [00:02<00:00, 8.67MB/s]

Unzipping dataset...
Creating target folders...
Sorting images...

=== SYSTEM REPORT ===
Chicken images loaded: 447
Duck images loaded: 310
SUCCESS! You are ready to run Cell 3 (Data Loading).


## Data Loading and Transformation

In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import classification_report
import numpy as np

# Define transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load data
dataset = datasets.ImageFolder(root='dataset', transform=transform)

# Split into train/val (80/20)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# VERIFICATION PRINTS:
print(f"Total images loaded: {len(dataset)}")
print(f"Classes found: {dataset.classes}")
print(f"Using device: {device}")

Total images loaded: 757
Classes found: ['chicken', 'duck']
Using device: cuda


## Fine-Tuning ResNet18

In [ ]:

# Load pre-trained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Freeze base layers
for param in model.parameters():
    param.requires_grad = False

# Replace the final fully connected layer
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2) # 2 classes: Chicken, Duck
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# Training Loop
epochs = 5
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f}")

Epoch 1/5 - Loss: 0.4738
Epoch 2/5 - Loss: 0.2429
Epoch 3/5 - Loss: 0.2003
Epoch 4/5 - Loss: 0.1780
Epoch 5/5 - Loss: 0.1453


## Evaluation & Classification Report

In [ ]:

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

class_names = dataset.classes
print("\nClassification Report (Image Transfer Learning):")
print(classification_report(all_labels, all_preds, target_names=class_names))


Classification Report (Image Transfer Learning):
              precision    recall  f1-score   support

     chicken       0.97      0.95      0.96        82
        duck       0.94      0.97      0.96        70

    accuracy                           0.96       152
   macro avg       0.96      0.96      0.96       152
weighted avg       0.96      0.96      0.96       152

